## PyTorch manual_seed and randomness reproducibility

Short version: `manual_seed` initializes the PRNG (Pytorch Random Number Generator) **state once**; every `torch.rand` call **advances** that state. So the same seed reproduces the same *sequence*, not the same *value on every call*.

```python
torch.manual_seed(0)
a = torch.rand(3)   # same every run
b = torch.rand(3)   # always differs from a, but same every run
```

If you reseed before each draw, you get identical tensors:

```python
torch.manual_seed(0); x1 = torch.rand(5)
torch.manual_seed(0); x2 = torch.rand(5)   # x1 == x2
```

### How it works

- CPU uses a Mersenne Twister; CUDA uses Philox, with an **independent generator per device**.
- `torch.manual_seed(s)` seeds the CPU generator and (in recent versions) all CUDA devices. `torch.cuda.manual_seed_all(s)` seeds CUDA only.
- The generator is global and stateful. Anything that draws randomness — model weight init, `dropout`, `shuffle=True`, `randperm`, DataLoader workers — consumes from it, shifting all subsequent results.
- NumPy (`np.random`) and Python's `random` have **separate** states; `manual_seed` does not touch them.


## Tensor factory / in-place random ops
- `torch.rand`, `randn`, `randint`, `randperm`, `*_like` variants
- `torch.normal`, `bernoulli`, `multinomial`, `poisson`
- In-place: `.uniform_`, `.normal_`, `.random_`, `.bernoulli_`, `.exponential_`, `.geometric_`, `.cauchy_`, `.log_normal_`

## Module *construction* (biggest silent consumer)
Every `nn.Linear/Conv2d/Embedding/LSTM/...` calls `reset_parameters()` → `nn.init.*` at `__init__`, drawing from the global RNG:

- `nn.init.uniform_, normal_, xavier_uniform_/normal_, kaiming_uniform_/normal_, orthogonal_, sparse_, trunc_normal_`
- `model.apply(custom_init)` and lazy modules on first forward
- `timm`/HF/hub model factories (many use their own init)

So `torch.manual_seed(0); model = Net(); torch.rand(3)` ≠ `torch.manual_seed(0); torch.rand(3)`.

## Modules in training mode only
- `nn.Dropout`, `Dropout2d/3d`, `AlphaDropout`, `FeatureAlphaDropout` (`F.dropout*`)
- `nn.RReLU` (`F.rrelu` with `training=True`)
- `F.gumbel_softmax`
- `nn.LSTM/GRU/RNN` with `dropout>0`
- `nn.MultiheadAttention` / Transformer layers with `dropout>0`
- `F.scaled_dot_product_attention(..., dropout_p>0)` while training
- Eval mode (`model.eval()`) → no draw for all of the above

## Data pipeline
- `DataLoader(shuffle=True)` → `RandomSampler` → `randperm` each epoch
- `WeightedRandomSampler`, `BatchSampler` with randomness
- **Creating a `DataLoader` iterator once draws a base seed** for workers (`random_()` on the loader's generator); each epoch re-iterating draws again
- `worker_init_fn`, `generator=` on DataLoader

## Distributions
- `torch.distributions.*.sample()` / `.rsample()` (constructor alone usually doesn't draw)
- `Categorical.sample()` → `multinomial`, `Normal.sample()` → `normal`

## Not PyTorch's RNG (separate states)
- `torchvision` PIL transforms use Python `random` (v2 tensor transforms use torch)
- `numpy`, PIL, `albumentations`, Python `random`
- torchaudio masking transforms (`FrequencyMasking`/`TimeMasking`)

## Does *not* advance
`torch.empty/zeros/ones/full/eye/arange/linspace/tensor`, `.fill_`, arithmetic, `clone`, `manual_seed` (resets), `get/set_rng_state`, `torch.quasirandom.SobolEngine` (own state).

Quick way to test any call yourself:

```python
def advances(fn, device="cpu"):
    s = torch.get_rng_state() if device=="cpu" else torch.cuda.get_rng_state(device)
    fn()
    s2 = torch.get_rng_state() if device=="cpu" else torch.cuda.get_rng_state(device)
    return not torch.equal(s, s2)
```